# Water Quality Data Processing

**Pipeline**
1. Merge common columns from *Dams & Lakes* and *Rivers* datasets (1999–2012)
2. Match each station with GPS coordinates from three reference files
3. Compute **monthly averages** per station across all measurement columns

**Output:** `water_quality_monthly_averages.xlsx`
- Sheet `Combined_Data` – merged raw data with GPS info
- Sheet `Monthly_Averages` – per-station monthly means

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

## Step 1 – Load Dams & Rivers data

In [ ]:
def build_column_names(raw_df):
    """
    Combine row-2 header names with units from row-1 to produce unique names.
    row 0 = section titles, row 1 = units, row 2 = actual column names.
    """
    units = raw_df.iloc[1].fillna("").astype(str)
    names = raw_df.iloc[2].fillna("").astype(str).str.strip()

    final = []
    seen = {}
    for i, (name, unit) in enumerate(zip(names, units)):
        if name == "" or name == "nan":
            name = f"col_{i}"
        key = name
        if key in seen:
            unit_clean = unit.strip().replace("/", "_").replace(" ", "")
            if unit_clean and unit_clean != "nan":
                candidate = f"{name}_{unit_clean}"
            else:
                candidate = f"{name}_{seen[key]}"
            seen[name] += 1
            seen[candidate] = 1
            final.append(candidate)
        else:
            seen[key] = 1
            final.append(name)
    return final


def load_data_file(filepath, source_label):
    """Load a water-quality data file, skip the two header rows above row 2."""
    raw = pd.read_excel(filepath, header=None, dtype=str)
    col_names = build_column_names(raw)

    df = raw.iloc[3:].copy()
    df.columns = col_names
    df = df.reset_index(drop=True)

    df.rename(columns={col_names[0]: "STATION_ID", col_names[1]: "POINT_ID"}, inplace=True)
    df["SOURCE"] = source_label
    return df


print("Loading Dams & Lakes data …")
dams_raw = load_data_file("Dams and lakes 1999-2012 (A-X).xlsx", "DAMS_LAKES")
print(f"  {len(dams_raw):,} rows loaded")

print("Loading Rivers data …")
rivers_raw = load_data_file("Rivers 1999-2012 (A-X).xlsx", "RIVERS")
print(f"  {len(rivers_raw):,} rows loaded")

## Step 2 – Keep only common columns

In [ ]:
dams_cols   = set(dams_raw.columns) - {"SOURCE"}
rivers_cols = set(rivers_raw.columns) - {"SOURCE"}
common_cols = sorted(dams_cols & rivers_cols)

print(f"Dams columns  : {len(dams_cols)}")
print(f"Rivers columns: {len(rivers_cols)}")
print(f"Common columns: {len(common_cols)}")
print("\nColumn list:", common_cols)

dams_df   = dams_raw[common_cols + ["SOURCE"]].copy()
rivers_df = rivers_raw[common_cols + ["SOURCE"]].copy()

combined = pd.concat([dams_df, rivers_df], ignore_index=True)
print(f"\nCombined rows: {len(combined):,}")

## Step 3 – Load station reference files & attach GPS

In [ ]:
GPS_COLS = [
    "SOUTH LAT. DECIMAL DEGR",
    "SOUTH LON. DECIMAL DEGR",
    "BRIEF LOCALITY DESCRIPTION OF SAMPLE STATION",
    "RIVER OR LAKE/DAM",
]


def normalize_station_df(filepath):
    df = pd.read_excel(filepath)
    df.columns = [c.replace("\n", " ").strip() for c in df.columns]
    rename = {}
    for c in df.columns:
        if "STATION ID" in c:
            rename[c] = "STATION_ID"
        elif c == "POINT ID":
            rename[c] = "POINT_ID"
    df.rename(columns=rename, inplace=True)
    keep = [c for c in ["STATION_ID", "POINT_ID"] + GPS_COLS if c in df.columns]
    return df[keep].copy()


print("Loading station reference files …")
st_with_id = normalize_station_df("Sample stations with ID.xlsx")
st_no_id   = normalize_station_df("Sample stations without ID.xlsx")
st_z_id    = normalize_station_df("Samples stations with Z-ID.xlsx")

all_stations = pd.concat([st_with_id, st_no_id, st_z_id], ignore_index=True)

station_id_lookup = (
    all_stations.dropna(subset=["STATION_ID"])
    .drop_duplicates(subset=["STATION_ID"])
    .set_index("STATION_ID")[GPS_COLS]
    .to_dict("index")
)

point_id_lookup = (
    all_stations.dropna(subset=["POINT_ID"])
    .drop_duplicates(subset=["POINT_ID"])
    .set_index("POINT_ID")[GPS_COLS]
    .to_dict("index")
)

print(f"Station-ID lookup entries: {len(station_id_lookup):,}")
print(f"Point-ID  lookup entries : {len(point_id_lookup):,}")

In [ ]:
def attach_gps(row):
    """Look up GPS by STATION_ID first, then POINT_ID."""
    sid = str(row["STATION_ID"]).strip() if pd.notna(row["STATION_ID"]) else ""
    pid = str(row["POINT_ID"]).strip()   if pd.notna(row["POINT_ID"])   else ""

    try:
        pid_num = int(float(pid))
    except (ValueError, TypeError):
        pid_num = None

    info = station_id_lookup.get(sid) or point_id_lookup.get(pid_num) or {}
    return pd.Series({
        "LAT":        info.get("SOUTH LAT. DECIMAL DEGR"),
        "LON":        info.get("SOUTH LON. DECIMAL DEGR"),
        "LOCATION":   info.get("BRIEF LOCALITY DESCRIPTION OF SAMPLE STATION"),
        "WATER_BODY": info.get("RIVER OR LAKE/DAM"),
    })


print("Attaching GPS coordinates …")
gps_cols = combined.apply(attach_gps, axis=1)
combined = pd.concat([combined, gps_cols], axis=1)

matched = combined["LAT"].notna().sum()
print(f"Rows with GPS match: {matched:,} / {len(combined):,}")

## Step 4 – Clean data & parse dates

In [ ]:
# Replace -9999 sentinel with NaN
combined.replace(-9999, np.nan, inplace=True)
combined.replace("-9999", np.nan, inplace=True)

# Parse DATE column
combined["DATE"]  = pd.to_datetime(combined["DATE"], errors="coerce")
combined["YEAR"]  = combined["DATE"].dt.year
combined["MONTH"] = combined["DATE"].dt.month

# Identify measurement columns
ID_COLS   = ["STATION_ID", "POINT_ID", "SOURCE", "DATE", "YEAR", "MONTH",
             "LAT", "LON", "LOCATION", "WATER_BODY"]
MEAS_COLS = [c for c in combined.columns if c not in ID_COLS]

for col in MEAS_COLS:
    combined[col] = pd.to_numeric(combined[col], errors="coerce")

print(f"Measurement columns ({len(MEAS_COLS)}):")
print(MEAS_COLS)

## Step 5 – Compute monthly averages per station

In [ ]:
GROUP_KEYS = ["STATION_ID", "YEAR", "MONTH"]

monthly_avg = (
    combined
    .groupby(GROUP_KEYS, dropna=False)[MEAS_COLS]
    .mean(numeric_only=True)
    .reset_index()
)

# Attach GPS info back
gps_ref = (
    combined[["STATION_ID", "POINT_ID", "LAT", "LON", "LOCATION", "WATER_BODY", "SOURCE"]]
    .drop_duplicates(subset=["STATION_ID"])
)
monthly_avg = monthly_avg.merge(gps_ref, on="STATION_ID", how="left")

# Reorder columns
front_cols = ["STATION_ID", "POINT_ID", "SOURCE", "WATER_BODY", "LOCATION",
              "LAT", "LON", "YEAR", "MONTH"]
remaining  = [c for c in monthly_avg.columns if c not in front_cols]
monthly_avg = monthly_avg[front_cols + remaining]

print(f"Monthly averages : {len(monthly_avg):,} rows × {len(monthly_avg.columns)} columns")
print(f"Unique stations  : {monthly_avg['STATION_ID'].nunique():,}")
monthly_avg.head()

## Step 6 – Write output Excel file

In [ ]:
OUT_FILE = "water_quality_monthly_averages.xlsx"

combined_out_cols = (
    ["STATION_ID", "POINT_ID", "SOURCE", "WATER_BODY", "LOCATION",
     "LAT", "LON", "DATE", "YEAR", "MONTH"]
    + MEAS_COLS
)
combined_out_cols = [c for c in combined_out_cols if c in combined.columns]

with pd.ExcelWriter(OUT_FILE, engine="openpyxl") as writer:
    combined[combined_out_cols].to_excel(writer, sheet_name="Combined_Data", index=False)
    monthly_avg.to_excel(writer, sheet_name="Monthly_Averages", index=False)

print(f"Output written to: {OUT_FILE}")
print(f"  Sheet 'Combined_Data'    : {len(combined):,} rows")
print(f"  Sheet 'Monthly_Averages' : {len(monthly_avg):,} rows")